# RL Locomotion - Training on Google Colab

PPO training of locomotion policies. This notebook runs the local `src/train.py` on a free Colab T4 GPU.

## Configuration

Edit the variables in the next cell, then `Runtime -> Run all`.


In [ ]:
# === EDIT ME ===
GITHUB_REPO   = "https://github.com/tuanwannafly/RL-Locomotion-Sim-to-Real-Robustness.git"
GITHUB_BRANCH = "main"

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    GITHUB_TOKEN = ""   # public repo: leave empty if no secret set

RUN_NAME    = "baseline_ant"
CONFIG      = "configs/baseline_ant.yaml"
ENV_ID      = "Ant-v4"
TIMESTEPS   = 2_500_000
DEVICE      = "cuda"

DRIVE_RUNS  = "/content/drive/MyDrive/rl-locomotion/runs"
REPO_DIR    = "/content/rl-locomotion"
SAVE_DIR    = f"{DRIVE_RUNS}/{RUN_NAME}"
print(f"Save dir: {SAVE_DIR}")


## 1. Mount Drive, clone repo, install deps

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, pathlib
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.chdir("/content")

if not pathlib.Path(REPO_DIR).exists():
    url = GITHUB_REPO
    if GITHUB_TOKEN:
        url = url.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--branch", GITHUB_BRANCH, url, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

os.chdir(REPO_DIR)
print("Repo at:", os.getcwd())
print(subprocess.check_output(["git", "log", "--oneline", "-3"]).decode())


In [ ]:
# install PyTorch (CUDA) + project deps
subprocess.check_call(["pip", "install", "-q",
    "torch==2.7.0", "--index-url", "https://download.pytorch.org/whl/cu118"])
subprocess.check_call(["pip", "install", "-q", "-r", "requirements.txt"])
print("deps installed")


In [ ]:
# quick import sanity-check (catches broken installs before the long train)
import gymnasium, stable_baselines3, torch, numpy
print("gymnasium", gymnasium.__version__, "| sb3", stable_baselines3.__version__,
      "| torch", torch.__version__, "| numpy", numpy.__version__)
print("cuda available:", torch.cuda.is_available())


## 2. Verify GPU

In [ ]:
!nvidia-smi -L || echo "(no GPU)"
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Run training

Calls `src/train.py` and streams logs into the notebook. Final model and TensorBoard events are saved to Drive.

In [ ]:
# Workaround: avoid native-lib segfaults seen when running train.py via
# `subprocess.run(["python", ...])` on Colab Python 3.12 (libomp/zmq).
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Import lazily so the kernel state (Drive mount, clone) is ready first.
from src.train import main as train_main

# Drive argv the way src/train.py expects it.
sys.argv = [
    "src/train.py",
    "--config",   CONFIG,
    "--env",      ENV_ID,
    "--timesteps", str(TIMESTEPS),
    "--run-name", RUN_NAME,
    "--save-dir", SAVE_DIR,
    "--device",   DEVICE,
]

# Streaming a long train (2.5M steps) live in the cell output is hard;
# if it dies early you will see a real Python traceback (not -11) now.
raise SystemExit(train_main())


## 4. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{DRIVE_RUNS}" --port 6006


## 5. Download artefacts to local

After the cell above finishes, go to `MyDrive/rl-locomotion/runs/<RUN_NAME>/` and download:

- `model.zip`         -> local `experiments/models/<RUN_NAME>/model.zip`
- `vecnormalize.pkl`  -> local `experiments/models/<RUN_NAME>/vecnormalize.pkl`
- `monitor.csv`       -> local `experiments/logs/<RUN_NAME>/monitor.csv`
- `tb/` (optional)    -> local `experiments/logs/<RUN_NAME>/tb/`

Then continue with Sprint 1 analysis on local.
